# 01.2 Tensor Shape Ops / 张量形状操作

这一节的核心任务是建立真正的形状思维 / shape thinking。  
The core task of this notebook is to build genuine shape thinking.

很多 `PyTorch` bug 不是数学错误，而是形状错误。  
Many `PyTorch` bugs are not mathematical errors, but shape errors.

重点概念 / Key concepts:

- 变形 / reshape
- 展平 / flatten
- 增减维度 / unsqueeze and squeeze
- 转置 / transpose
- 维度重排 / permute
- 广播 / broadcasting

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 熟练使用 `reshape`、`view`、`flatten` / Use `reshape`, `view`, and `flatten` comfortably.
2. 理解 `unsqueeze` 和 `squeeze` / Understand `unsqueeze` and `squeeze`.
3. 区分 `transpose` 和 `permute` / Distinguish `transpose` from `permute`.
4. 预测常见操作后的输出形状 / Predict output shapes after common operations.
5. 用 shape 解释广播 / Explain broadcasting using shapes.
6. 更快定位 shape mismatch 报错 / Debug shape mismatch errors faster.

In [ ]:
import torch

## 1. `reshape`、`view` 与 `flatten`
## `reshape`, `view`, and `flatten`

它们都和“张量外形如何变化”有关。  
They all describe how the external shape of a tensor changes.

- `reshape`：最常用，比较稳妥 / the most commonly used and usually convenient
- `view`：要求张量内存布局满足条件 / requires compatible memory layout
- `flatten`：把若干维压平 / collapses dimensions into one

In [ ]:
x = torch.arange(24)
a = x.reshape(2, 3, 4)
b = a.flatten()
c = a.flatten(start_dim=1)

print("x.shape =", x.shape)
print("a.shape =", a.shape)
print("b.shape =", b.shape)
print("c.shape =", c.shape)

`flatten(start_dim=1)` 在神经网络里很常见。  
`flatten(start_dim=1)` is very common in neural networks.

例如 / For example:

- 输入是 `(batch, channels, height, width)`
- 展平后变成 `(batch, features)`

这通常发生在卷积层接全连接层之前。  
This usually happens before feeding convolution outputs into linear layers.

In [ ]:
# 练习 1 / Exercise 1
# 已知 x.shape == (2, 3, 4)
# Given x.shape == (2, 3, 4)
#
# 1. 把它 reshape 成 (6, 4)
# 2. 把它 flatten 成 (2, 12)

x = torch.arange(24).reshape(2, 3, 4)

# x1 =
# x2 =
# print(x1.shape)
# print(x2.shape)

In [ ]:
# 练习 1 参考答案 / Exercise 1 Reference Solution

x = torch.arange(24).reshape(2, 3, 4)
x1 = x.reshape(6, 4)
x2 = x.flatten(start_dim=1)
print(x1.shape)
print(x2.shape)

## 2. `unsqueeze` 与 `squeeze`
## `unsqueeze` and `squeeze`

这两个操作的本质是“增加或删除长度为 1 的维度”。  
These two operations add or remove dimensions of size 1.

- `unsqueeze(dim)`：在某个位置插入一个长度为 1 的维度 / insert a size-1 dimension
- `squeeze(dim)`：删除长度为 1 的维度 / remove a size-1 dimension

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
x_row = x.unsqueeze(0)
x_col = x.unsqueeze(1)
x_back = x_col.squeeze(1)

print("x.shape =", x.shape)
print("x_row.shape =", x_row.shape)
print("x_col.shape =", x_col.shape)
print("x_back.shape =", x_back.shape)

这些操作常见于：  
These operations commonly appear when:

- 手工构造 batch 维 / manually creating a batch dimension
- 调整广播方向 / controlling broadcasting direction
- 让输入符合层的接口要求 / matching a layer's input interface

In [ ]:
# 练习 2 / Exercise 2
# 给定 x.shape == (4,)
# Given x.shape == (4,)
#
# 1. 变成 (1, 4)
# 2. 变成 (4, 1)
# 3. 再把 (4, 1) 变回 (4,)

x = torch.arange(4)

# a =
# b =
# c =
# print(a.shape, b.shape, c.shape)

In [ ]:
# 练习 2 参考答案 / Exercise 2 Reference Solution

x = torch.arange(4)
a = x.unsqueeze(0)
b = x.unsqueeze(1)
c = b.squeeze(1)
print(a.shape, b.shape, c.shape)

## 3. `transpose` 与 `permute`
## `transpose` and `permute`

这两个操作都会调整维度顺序，但粒度不同。  
Both operations reorder dimensions, but at different levels of flexibility.

- `transpose(dim0, dim1)`：交换两个维度 / swap two dimensions
- `permute(...)`：按给定顺序重排所有维度 / reorder all dimensions by a specified order

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
xt = x.transpose(1, 2)
xp = x.permute(2, 0, 1)

print("x.shape =", x.shape)
print("xt.shape =", xt.shape)
print("xp.shape =", xp.shape)

图像任务里常见形状 / Common image shape pattern:

- `PyTorch` 常用 `(batch, channels, height, width)`
- 某些外部库常见 `(height, width, channels)`

这就是 `permute` 经常出现的原因之一。  
This is one reason why `permute` appears so often.

In [ ]:
# 练习 3 / Exercise 3
# 已知 image.shape == (3, 32, 32)，表示 (channels, height, width)
# Given image.shape == (3, 32, 32), meaning (channels, height, width)
#
# 请转成 (height, width, channels)
# Convert it to (height, width, channels)

image = torch.randn(3, 32, 32)

# image_hwc =
# print(image_hwc.shape)

In [ ]:
# 练习 3 参考答案 / Exercise 3 Reference Solution

image = torch.randn(3, 32, 32)
image_hwc = image.permute(1, 2, 0)
print(image_hwc.shape)

## 4. 广播 / Broadcasting

广播的核心不是“神奇地自动扩展”，而是按维度对齐。  
Broadcasting is not magic auto-expansion; it is dimension alignment.

最小规则 / Minimal rule:

- 从最后一维开始看 / compare dimensions from the end
- 两边相等，或某一边为 1 / dimensions must match, or one of them must be 1

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
offset = torch.tensor([10.0, 100.0])
row_scale = torch.tensor([[1.0], [10.0], [100.0]])

print("x + offset =\n", x + offset)
print()
print("x * row_scale =\n", x * row_scale)

理解方式 / How to read it:

- `x.shape == (3, 2)`
- `offset.shape == (2,)`，对齐最后一维 / aligns with the last dimension
- `row_scale.shape == (3, 1)`，按行广播 / broadcasts row-wise

In [ ]:
# 练习 4 / Exercise 4
# 实现 center_by_column(x)：让每一列减去自己的均值。
# Implement center_by_column(x) so each column subtracts its own mean.

def center_by_column(x):
    # TODO
    pass


# sample = torch.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
# print(center_by_column(sample))

In [ ]:
# 练习 4 参考答案 / Exercise 4 Reference Solution

def center_by_column_solution(x):
    col_mean = x.mean(dim=0, keepdim=True)
    return x - col_mean


sample = torch.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
print(center_by_column_solution(sample))

## 5. 非连续张量 / Non-Contiguous Tensors

这是一个非常实用但容易被忽略的点。  
This is a practical but often overlooked topic.

某些操作如 `transpose` 或 `permute` 后，张量可能不是连续的 / contiguous。  
After operations like `transpose` or `permute`, a tensor may become non-contiguous.

这时 / In such cases:

- `reshape` 通常更稳妥 / `reshape` is usually safer
- `view` 可能报错 / `view` may fail

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
xt = x.transpose(1, 2)

print("xt.is_contiguous() =", xt.is_contiguous())

try:
    bad = xt.view(2, 12)
    print("view result shape =", bad.shape)
except RuntimeError as e:
    print("view failed / view 失败:", e)

good = xt.reshape(2, 12)
print("reshape result shape =", good.shape)

In [ ]:
# 练习 5 / Exercise 5
# 给定 x.shape == (2, 3, 4)
# Given x.shape == (2, 3, 4)
#
# 1. 先 transpose 成 (2, 4, 3)
# 2. 再 reshape 成 (2, 12)

x = torch.arange(24).reshape(2, 3, 4)

# y =
# z =
# print(y.shape)
# print(z.shape)

In [ ]:
# 练习 5 参考答案 / Exercise 5 Reference Solution

x = torch.arange(24).reshape(2, 3, 4)
y = x.transpose(1, 2)
z = y.reshape(2, 12)
print(y.shape)
print(z.shape)

## 6. 小结 / Summary

你现在应该逐渐形成一个固定习惯：  
You should now start building a fixed habit:

1. 先写出输入 shape / first write down the input shape
2. 再预测输出 shape / then predict the output shape
3. 最后再运行代码验证 / finally run the code to verify

你现在应该能回答 / You should now be able to answer:

1. `reshape`、`flatten`、`unsqueeze` 各自解决什么问题？ / What problems do `reshape`, `flatten`, and `unsqueeze` solve?
2. `transpose` 和 `permute` 的差别是什么？ / What is the difference between `transpose` and `permute`?
3. 为什么 `view` 有时会失败，而 `reshape` 可以工作？ / Why does `view` sometimes fail while `reshape` works?
4. 广播的最小判断规则是什么？ / What is the minimal rule for broadcasting?

下一步建议 / Suggested next step:

- 进入 `01_03_autograd.ipynb`，理解梯度是怎么被记录和传播的 / Move to `01_03_autograd.ipynb` to understand how gradients are tracked and propagated.